In [1]:
import os
import pyspark

spark_version = pyspark.__version__
print(f"PySpark version: {spark_version}")

if spark_version.startswith("4"):
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
else:
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"

os.environ['PYSPARK_SUBMIT_ARGS'] = f'--packages {KAFKA_PACKAGE} pyspark-shell'
print(f"Connector: {KAFKA_PACKAGE}")

PySpark version: 4.0.0.dev2
Connector: org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2


In [2]:
"""
Initialize Spark Session with the Kafka connector.
The connector is downloaded from Maven Central on first run.
"""
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CryptoMonitor")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .config("spark.sql.shuffle.partitions", "2")  # small workload, reduce overhead
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — ready")

Spark 4.0.0-preview2 — ready


In [3]:
"""
Subscribe to the 'prices' topic populated by producer.py.
startingOffsets='latest' = only new messages, no historical replay.
"""
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "prices")
    .option("startingOffsets", "latest")
    .load()
)
kafka_raw.printSchema()  # Kafka envelope: key, value, topic, partition, offset, ...

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [4]:
"""
Decode Kafka payload: bytes -> UTF-8 string -> JSON struct -> flat columns.
Convert ISO 8601 timestamp string to TimestampType (required for windows).
"""
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

# Schema must match producer.py output
price_schema = StructType([
    StructField("symbol",     StringType()),
    StructField("price",      DoubleType()),
    StructField("quantity",   DoubleType()),
    StructField("trade_time", LongType()),
    StructField("timestamp",  StringType()),
])

df = (
    kafka_raw
    .select(from_json(col("value").cast("string"), price_schema).alias("tx"))
    .select("tx.*")
    .withColumn("event_time", to_timestamp("timestamp"))
)
df.printSchema()

root
 |-- symbol: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: double (nullable = true)
 |-- trade_time: long (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [5]:
"""
Inspect parsed records to verify schema. Stops after 2 micro-batches.
"""
batch_counter = {"n": 0}

def show_parsed(df, batch_id):
    batch_counter["n"] += 1
    print(f"\n--- Batch {batch_id} ---")
    df.show(5, truncate=False)
    if batch_counter["n"] >= 2:
        raise Exception("stop")  # force exit after 2 batches

q = (
    df.writeStream
    .foreachBatch(show_parsed)
    .option("checkpointLocation", "/tmp/chk_crypto_parsed")
    .start()
)
try:
    q.awaitTermination()
except Exception:
    q.stop()
print("Stopped.")


--- Batch 1 ---
+-------+-------+--------+-------------+--------------------------+--------------------------+
|symbol |price  |quantity|trade_time   |timestamp                 |event_time                |
+-------+-------+--------+-------------+--------------------------+--------------------------+
|ETHUSDT|1902.36|0.0032  |1780433172909|2026-06-02T22:46:13.170874|2026-06-02 22:46:13.170874|
+-------+-------+--------+-------------+--------------------------+--------------------------+


--- Batch 2 ---
+-------+--------+--------+-------------+--------------------------+--------------------------+
|symbol |price   |quantity|trade_time   |timestamp                 |event_time                |
+-------+--------+--------+-------------+--------------------------+--------------------------+
|ETHUSDT|1902.35 |0.777   |1780433174223|2026-06-02T22:46:14.487275|2026-06-02 22:46:14.487275|
|ETHUSDT|1902.35 |0.8     |1780433174223|2026-06-02T22:46:14.488156|2026-06-02 22:46:14.488156|
|ETHUSDT|1

In [6]:
"""
Compute per-symbol stats over 1-minute tumbling windows.
Volatility = standard deviation of price within the window.
Watermark of 30s tells Spark how long to wait for late events.
"""
from pyspark.sql.functions import (
    window, count, avg, stddev, max as _max, min as _min, round as _round
)

windowed_stats = (
    df
    .withWatermark("event_time", "30 seconds")
    .groupBy(window("event_time", "1 minute"), "symbol")
    .agg(
        count("price").alias("trades"),
        _round(avg("price"), 2).alias("avg_price"),
        _round(stddev("price"), 4).alias("volatility"),  # key signal for anomaly
        _round(_min("price"), 2).alias("min_price"),
        _round(_max("price"), 2).alias("max_price"),
    )
)
windowed_stats.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- symbol: string (nullable = true)
 |-- trades: long (nullable = false)
 |-- avg_price: double (nullable = true)
 |-- volatility: double (nullable = true)
 |-- min_price: double (nullable = true)
 |-- max_price: double (nullable = true)



In [7]:
"""
Print closed windows as a formatted table.
First ~90s shows '(no closed windows yet)' — watermark hasn't advanced.
Auto-stops after 8 batches.
"""
batch_counter["n"] = 0

def show_windowed(df, batch_id):
    batch_counter["n"] += 1
    print(f"\n=== Batch {batch_id} ===")
    rows = df.select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "symbol", "trades", "avg_price",
        "volatility", "min_price", "max_price",
    ).orderBy("from", "symbol").collect()

    if not rows:
        print("(no closed windows yet — waiting for watermark to advance)")
        return

    for r in rows:
        print(f"{r['from'].strftime('%H:%M')}-{r['to'].strftime('%H:%M')}  "
              f"{r['symbol']:>8s}  "
              f"trades={r['trades']:>4d}  "
              f"avg={r['avg_price']:>10.2f}  "
              f"vol={r['volatility'] or 0:>8.4f}  "
              f"range=[{r['min_price']:.2f}, {r['max_price']:.2f}]")

    if batch_counter["n"] >= 8:
        raise Exception("stop")

q = (
    windowed_stats.writeStream
    .outputMode("append")          # emit each window once when it closes
    .foreachBatch(show_windowed)
    .option("checkpointLocation", "/tmp/chk_crypto_windows")
    .trigger(processingTime="20 seconds")
    .start()
)
try:
    q.awaitTermination()
except Exception:
    q.stop()
print("Stopped.")

Stopped.


In [8]:
"""
Filter windows with high volatility and publish to the 'alerts' topic.
Rule-based thresholds (Day 3 will replace these with an ML model).
"""
from pyspark.sql.functions import to_json, struct, lit

alerts = (
    windowed_stats
    .filter(
    ((col("symbol") == "BTCUSDT") & (col("volatility") > 10)) |
    ((col("symbol") == "ETHUSDT") & (col("volatility") > 0.3))
)
    .select(
        # Kafka sink requires a single 'value' column — serialize struct to JSON
        to_json(
            struct(
                col("window.start").cast("string").alias("window_start"),
                col("window.end").cast("string").alias("window_end"),
                "symbol", "trades", "avg_price",
                "volatility", "min_price", "max_price",
                lit("HIGH_VOLATILITY").alias("alert_type"),
                lit("rule_based").alias("source"),
            )
        ).alias("value")
    )
)

alert_query = (
    alerts.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("topic", "alerts")
    .option("checkpointLocation", "/tmp/chk_crypto_alerts")
    .outputMode("append")
    .start()
)
print("Alert stream started. Will run for 90 seconds.\n")

import time
time.sleep(300)
alert_query.stop()
print("Alert query stopped.")

Alert stream started. Will run for 90 seconds.

Alert query stopped.


In [9]:
"""
Read back from the 'alerts' topic to confirm publishing worked.
count=0 is possible if the market was calm — lower thresholds in Cell 8 to test.
"""
from kafka import KafkaConsumer
import json

c = KafkaConsumer(
    'alerts',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    consumer_timeout_ms=5000,
)

count = 0
for msg in c:
    data = json.loads(msg.value.decode())
    print(f"{data['symbol']}  vol={data['volatility']:.4f}  window={data['window_start']}")
    count += 1
print(f"\nTotal alerts: {count}")

ETHUSDT  vol=0.5234  window=2026-06-02 22:50:00
BTCUSDT  vol=33.4115  window=2026-06-02 22:51:00
ETHUSDT  vol=0.3022  window=2026-06-02 22:56:00
ETHUSDT  vol=0.3347  window=2026-06-02 22:59:00
ETHUSDT  vol=0.4159  window=2026-06-02 22:58:00
BTCUSDT  vol=18.3111  window=2026-06-02 22:58:00
ETHUSDT  vol=0.6652  window=2026-06-02 22:53:00
ETHUSDT  vol=0.4741  window=2026-06-02 22:51:00
BTCUSDT  vol=20.9216  window=2026-06-02 22:53:00
ETHUSDT  vol=0.7268  window=2026-06-02 22:52:00
ETHUSDT  vol=0.5279  window=2026-06-02 22:57:00
BTCUSDT  vol=21.4203  window=2026-06-02 22:54:00
BTCUSDT  vol=28.8588  window=2026-06-02 22:52:00
ETHUSDT  vol=0.3330  window=2026-06-02 22:54:00
ETHUSDT  vol=0.3894  window=2026-06-02 22:55:00
BTCUSDT  vol=20.8505  window=2026-06-02 23:01:00
ETHUSDT  vol=0.4822  window=2026-06-02 23:01:00
ETHUSDT  vol=0.3898  window=2026-06-02 23:02:00
BTCUSDT  vol=27.2185  window=2026-06-02 23:03:00
ETHUSDT  vol=1.2313  window=2026-06-02 23:03:00
ETHUSDT  vol=0.3185  window=2026-

In [10]:
"""Release Spark resources before moving on to the next notebook."""
spark.stop()
print("Spark stopped.")

Spark stopped.
